# Task 14, Notebook 11: Fairness of the explanations, validation, manual review and the exported report

**Programme:** Aura Workforce by Growmar, AI/ML track, CortexHR **Phase 3: Intelligence**  
**Task:** 14: Explain every person's risk  
**Objective:** Check the risk report for group effects and wording problems, validate it against the sample format, read ten rows aloud and save the final 430-row CSV.  
**Workflow stages covered in this notebook:** 32, 33, 34, 35, 36, 37, 38 of the 38-stage Phase 1/2 standard.  
**Rule for every number below:** it is printed by the code in the same notebook; nothing is typed in by hand.

In [1]:
import sys, warnings, pathlib
warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'config' / 'phase3_config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
from cortexhr.phase3.common.config import load_config
from cortexhr.phase3.common import paths as P
CFG = load_config(); SEED = CFG['seed']
print('project root:', ROOT.name, '| seed:', SEED)

project root: phase3 | seed: 42


In [2]:
import joblib
from cortexhr.phase3.attrition import v2 as V
from cortexhr.phase3.common.io import read_csv, save_table, save_json
from cortexhr.phase3.common.validation import profile
from cortexhr.phase3.explanations import risk_report as RR
from cortexhr.phase3.explanations.groups import GROUPS, SENSITIVE_PROXY, check_groups
from cortexhr.phase3.explanations.reasons import sentence, wording_problems
from cortexhr.phase3.explanations.shapley import shapley_values, additivity_error, stability
scoring_raw = read_csv('attrition_scoring_2026'); sample = read_csv('sample_attrition_risk'); train = V.load_training()
print('scoring rows:', len(scoring_raw), '| sample rows:', len(sample))

scoring rows: 430 | sample rows: 430


In [3]:
scoring, X, join_rep = V.load_scoring(); rep = read_csv('sample_attrition_risk').iloc[0:0]; rep = pd.read_csv(P.PREDICTIONS / 'attrition_risk_2026.csv'); contrib = pd.read_csv(P.task_dir(14) / 'group_contributions.csv')
phi = contrib.drop(columns=['employee_id', 'base_value', 'prediction']); print('report rows:', len(rep), '| contributions:', phi.shape)

report rows: 430 | contributions: (430, 23)


## Stage 32: Fairness and Bias Evaluation

*What this stage is for:* Gaps between groups, measured with the protected attributes only after the fact.

Group effects. Gender and age come from the restricted attribute file and are used only to *measure* here. Coverage is checked first; then the band share and the explanation content by group.

In [4]:
from cortexhr.phase3.attrition.train import vault_attributes
va = vault_attributes()
if va is None: print('restricted file not present: group view skipped'); grp = None
else:
    a = va.drop_duplicates('person_ref').set_index('person_ref'); grp = rep.join(a[['gender', 'age_band']], on='employee_id'); print('employees with attribute record: %d of %d' % (grp.gender.notna().sum(), len(grp)))
    t = grp.groupby('gender').agg(n=('employee_id', 'size'), mean_risk=('risk_score_12m', 'mean'), high_share=('risk_band', lambda s: (s == 'High').mean()), medium_or_high=('risk_band', lambda s: (s != 'Low').mean())).round(3); print(t.to_string())
    t2 = grp.groupby('age_band').agg(n=('employee_id', 'size'), mean_risk=('risk_score_12m', 'mean'), high_share=('risk_band', lambda s: (s == 'High').mean())).round(3); print(t2.to_string())

employees with attribute record: 430 of 430
          n  mean_risk  high_share  medium_or_high
gender                                            
Female  183      0.121       0.098           0.290
Male    235      0.132       0.102           0.315
Other    12      0.125       0.083           0.167
            n  mean_risk  high_share
age_band                            
25-34     274      0.133       0.102
35-44     107      0.111       0.075
45+         4      0.162       0.250
<25        45      0.129       0.133


In [5]:
if grp is not None:
    tenure = scoring.set_index('employee_id').tenure_months.reindex(rep.employee_id).to_numpy(); bins = pd.cut(tenure, [-1, 11, 35, 10 ** 4], labels=['under 1 year', '1 to 3 years', 'over 3 years'])
    allr = rep[['top_reason_1', 'top_reason_2', 'top_reason_3']].astype(str).agg(' '.join, axis=1); ment = allr.str.contains('time at the company|length of service', regex=True)
    print(pd.DataFrame({'employees': pd.Series(bins).value_counts(), 'share_High': rep.groupby(bins, observed=True).risk_band.apply(lambda s: (s == 'High').mean()), 'tenure_in_reasons': ment.groupby(bins, observed=True).mean()}).round(3).to_string())
    print('Reading: employees with under one year of service are over-represented in High because the model learned that early tenure carries more risk in the history. The wording says so and says it is a pattern, not the person\'s performance.')

              employees  share_High  tenure_in_reasons
under 1 year         97       0.165              0.979
1 to 3 years        123       0.106              0.626
over 3 years        210       0.067              0.005
Reading: employees with under one year of service are over-represented in High because the model learned that early tenure carries more risk in the history. The wording says so and says it is a pattern, not the person's performance.


In [6]:
# how often would a sensitive-proxy group have been among the three largest contributions, had we shown it?
absrank = phi.abs().rank(axis=1, ascending=False); share = {g: float((absrank[g] <= 3).mean()) for g in SENSITIVE_PROXY}; print('share of employees where a sensitive-proxy group is in the top-3 by absolute contribution (hidden from the reasons):', {k: round(v, 3) for k, v in share.items()})

share of employees where a sensitive-proxy group is in the top-3 by absolute contribution (hidden from the reasons): {'work_location': 0.026, 'education': 0.044, 'career_history': 0.177}


## Stage 33: Model Calibration or Prediction Reliability

*What this stage is for:* Are the numbers trustworthy as numbers?

Reliability of the numbers. (1) the score is a calibrated probability (Task 13: ECE 0.014); (2) band edges are rank-based, so people near an edge could fall either side: how many are within one percentage point of the High cut-off?

In [7]:
cut = rep.loc[rep.risk_band == 'High', 'risk_score_12m'].min(); near = ((rep.risk_score_12m - cut).abs() <= 0.01).sum()
print('lowest High score %.3f | employees within 0.01 of that cut-off: %d of 430 | calibrated scores are probabilities; bands are rank cut-offs (top 10%% / next 20%%), not risk guarantees.' % (cut, near))

lowest High score 0.266 | employees within 0.01 of that cut-off: 5 of 430 | calibrated scores are probabilities; bands are rank cut-offs (top 10% / next 20%), not risk guarantees.


## Stage 34: Model Comparison and Final Selection

*What this stage is for:* Choose on evidence, not on the highest score alone.

attrition-v2 versus the simple baseline: do they flag the same people? (Task 13 concluded the models are equivalent in accuracy.)

In [8]:
from scipy.stats import spearmanr
d = V.load_training(); lr = V.baseline_models()['Logistic Regression (balanced)'].fit(d['X'], d['y']); pl = lr.predict_proba(X)[:, 1]
v2s = rep.risk_score_12m.to_numpy(); k = 43; a_, b_ = set(np.argsort(-v2s)[:k]), set(np.argsort(-pl)[:k])
print('rank correlation v2 vs baseline LR: %.3f | overlap of the 43 highest-risk people: %d of %d' % (spearmanr(v2s, pl)[0], len(a_ & b_), k))

rank correlation v2 vs baseline LR: 0.953 | overlap of the 43 highest-risk people: 38 of 43


## Stage 35: Final Prediction Generation

*What this stage is for:* Produce the deliverable.

The deliverable: the 430-row CSV in the sample's exact format (written by the explanation run, re-read here).

In [9]:
save_table(rep, P.task_dir(14) / 'attrition_risk_2026.csv'); rep.shape

(430, 11)

## Stage 36: Output and Business Validation

*What this stage is for:* Check the output against the required format and read it as the user would.

Validation against the sample format, then a manual read of ten rows.

In [10]:
chk = pd.DataFrame(RR.validate_report(rep, read_csv('sample_attrition_risk'))); print('checks passed: %d of %d' % (chk.passed.sum(), len(chk))); chk[['check', 'passed']]

checks passed: 14 of 14


                                                     check  passed
0                                                 430 rows    True
1                   columns equal sample (names and order)    True
2                                      employee ids unique    True
3            employee ids and order equal the scoring file    True
4                              risk score in [0,1], no NaN    True
5                                   risk band values valid    True
6                band counts 43/86/301 (top 10%, next 20%)    True
7                                band is monotone in score    True
8                          three non-empty reasons per row    True
9                          the three reasons are different    True
10                 every row says human review is required    True
11                           model_version is attrition-v2    True
12                              same dtype kinds as sample    True
13  no banned wording (certainty, blame, protected traits)    

In [11]:
rng = np.random.default_rng(SEED); ten = pd.concat([rep[rep.risk_band == 'High'].sample(3, random_state=1), rep[rep.risk_band == 'Medium'].sample(3, random_state=2), rep[rep.risk_band == 'Low'].sample(2, random_state=3),
                 rep.assign(t=scoring.set_index('employee_id').tenure_months.reindex(rep.employee_id).to_numpy()).sort_values('t').head(2).drop(columns='t')]).drop_duplicates('employee_id')
for _, r in ten.iterrows():
    print(f"{r.employee_id}  {r.risk_band:6s} score {r.risk_score_12m:.3f}")
    for k in (1, 2, 3): print(f"   {k}. {r[f'top_reason_{k}']}")
save_table(ten, P.task_dir(14) / 'manual_review_10_rows.csv')

EMP-0285  High   score 0.292
   1. The latest engagement score (2.3 out of 5) raised the predicted risk.
   2. The number of manager changes in the last 12 months (1) raised the predicted risk.
   3. Pay compared with the role's mid-point (0.62 times the mid-point) raised the predicted risk.
EMP-0266  High   score 0.388
   1. The latest engagement score (2.3 out of 5) raised the predicted risk.
   2. The number of manager changes in the last 12 months (1) raised the predicted risk.
   3. The role the person holds raised the predicted risk; this describes common patterns in the role, not the individual.
EMP-0597  High   score 0.302
   1. The department's attrition rate over the previous 12 months (5%) raised the predicted risk; this describes the team, not the individual.
   2. The latest engagement score (2.3 out of 5) raised the predicted risk.
   3. The length of service (1.4 years) raised the predicted risk; this reflects a common pattern at this stage of service, not the person's p

PosixPath('/home/claude/pack2/cortexhr/phase3/reports/task14/manual_review_10_rows.csv')

### Manual review: what reading ten rows aloud showed, and what was changed

I read these rows as an HR colleague would. First version of the wording, and the problems found:

1. **"Factors in the employee's career history raised the predicted risk (a general pattern ...)"** appeared in many rows. It is vague, HR cannot act on it, and career history can act as an age proxy. **Change:** groups that can proxy for protected traits (`career_history`, `education`, `work_location`) are no longer shown as reasons; they still count in the score, and the model note says so on every row.
2. **"The time at the company so far (151 months)"**: months are hard to read for long service, and the sentence could sound like a judgement of new joiners. **Change:** years for 12 months or more, "A short time at the company so far" under 12 months, and the tail *"this reflects a common pattern at this stage of service, not the person's performance"*. For new joiners the sentence therefore describes a pattern in the data, not a fault.
3. **"A missing recent performance rating lowered the predicted risk."** was confusing and read like a judgement. **Change:** "No recent performance rating is on record (a data gap, not a judgement)".
4. **High-risk people sometimes had a *protective* factor as reason 1** ("... lowered the predicted risk"). HR starts the conversation from what raised the score. **Change:** the three reasons are now the three factors that **raised** the score the most compared with an average employee (largest positive contribution); protective factors stay in `group_contributions.csv` for audit.
5. **"The time since the last raise (0 months)"** for people who just joined was odd. **Change:** "the time since the last raise or joining".

Remaining wording that is acceptable but not perfect: "The role the person holds raised the predicted risk" is generic; it is kept with the tail "common patterns in the role, not the individual". The 10 rows above are the re-read after these changes: each reason is a full sentence, names the employee's own value where it is safe to, never says the person will leave, and says on the row that human review is required.

**Think for yourself: how should tenure be described to a new joiner?** As a pattern learned from historical data, not as a judgement of the employee or proof that they will leave. For under one year of service, the report says 'A short time at the company so far' and explicitly adds that this reflects a common pattern at this stage of service, not the person's performance.

In [12]:
bad = [(r.employee_id, w) for _, r in rep.iterrows() for w in wording_problems(' '.join(r[['top_reason_1', 'top_reason_2', 'top_reason_3']].astype(str)))]
print('banned phrases found in all 1,290 reasons:', bad or 'none'); L = rep[['top_reason_1', 'top_reason_2', 'top_reason_3']].stack().str.len(); print('reason length (characters): median %d, max %d' % (L.median(), L.max()))
print('distinct reason sentences:', rep[['top_reason_1', 'top_reason_2', 'top_reason_3']].stack().nunique(), '| every row has the human-review note:', bool(rep.model_note.str.contains('Human review required').all()))

banned phrases found in all 1,290 reasons: none
reason length (characters): median 92, max 160
distinct reason sentences: 225 | every row has the human-review note: True


## Stage 37: Model Saving, Versioning and Reproducibility

*What this stage is for:* Artifacts, metadata, seeds, environment.

Artifacts: the CSV in two places, the contributions table, the run metadata, the model checksum.

In [13]:
import json; run = json.load(open(P.task_dir(14) / 'explanation_run.json')); print({k: run[k] for k in ('created_utc', 'n_permutations', 'background_rows', 'additivity_max_error')}); print('stability:', run['stability'])
print('model checksum recorded in Task 13 metadata:', json.load(open(P.METADATA / 'attrition_v2_metadata.json'))['artifact_sha256']['attrition_v2.joblib'][:16], '| CSV copies identical:', (P.PREDICTIONS / 'attrition_risk_2026.csv').read_bytes() == (P.task_dir(14) / 'attrition_risk_2026.csv').read_bytes())

{'created_utc': '2026-11-05T02:24:57+00:00', 'n_permutations': 120, 'background_rows': 200, 'additivity_max_error': 2.0816681711721685e-16}
stability: {'phi_correlation': 0.9714132901036352, 'top_n_identical_share': 0.4906976744186046, 'mean_top_n_overlap': 0.8178294573643411}
model checksum recorded in Task 13 metadata: c19d90438040800e | CSV copies identical: True


## Stage 38: Final Findings, Limitations and Conclusion

*What this stage is for:* What was found, what remains, what must not be done with it.

In [14]:
print('FINDINGS'); print('- 430 employees scored with attrition-v2; bands %s; mean risk %.3f.' % (rep.risk_band.value_counts().to_dict(), rep.risk_score_12m.mean()))
print('- Every row has three plain-English reasons, a human-review note and passes all %d format checks.' % len(chk)); print('- Explanation is exact in the efficiency sense (error %.1e) and moderately stable between independent runs (correlation %.2f); the order of the three reasons is the least reliable part.' % (run['additivity_max_error'], run['stability']['phi_correlation']))
print('LIMITS: model accuracy is modest (ROC-AUC about 0.70), reasons are associations in historical data, not causes; sensitive-proxy groups are hidden from reasons but still influence scores; for people already in the training history the score may be a little more confident; the 12-month outcome is unknown; HR must not use the list for dismissal, pay or promotion decisions.')

FINDINGS
- 430 employees scored with attrition-v2; bands {'Low': 301, 'Medium': 86, 'High': 43}; mean risk 0.127.
- Every row has three plain-English reasons, a human-review note and passes all 14 format checks.
- Explanation is exact in the efficiency sense (error 2.1e-16) and moderately stable between independent runs (correlation 0.97); the order of the three reasons is the least reliable part.
LIMITS: model accuracy is modest (ROC-AUC about 0.70), reasons are associations in historical data, not causes; sensitive-proxy groups are hidden from reasons but still influence scores; for people already in the training history the score may be a little more confident; the 12-month outcome is unknown; HR must not use the list for dismissal, pay or promotion decisions.
